In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 1: Imports
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import time
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    classification_report, confusion_matrix,
    roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm

# ──────────────────────────────────────────────────────────────────────────────
# CELL 2: Configuration — Kaggle
# ──────────────────────────────────────────────────────────────────────────────
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────────
    BASE_PATH = "/kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Preprocessed_dataset"

    TRAIN_CSV = "/kaggle/working/shuffled_csvs/train_shuffled.csv"
    VAL_CSV   = "/kaggle/working/shuffled_csvs/val_shuffled.csv"
    TEST_CSV  = "/kaggle/working/shuffled_csvs/test_shuffled.csv"

    IMAGE_ROOT = ""

    # Base checkpoint dir; each seed gets its own subdir at runtime
    CHECKPOINT_DIR = "/kaggle/working/TAL_GRN_checkpoints"

    IMG_COL   = "image_path"
    LABEL_COL = "label"

    # ── Model ──────────────────────────────────────────────────────────────────
    BACKBONE        = "resnet50"
    FEATURE_DIM     = 2048
    GNN_HIDDEN_DIM  = 512
    GNN_LAYERS      = 3
    NUM_CLASSES     = None

    # ── Graph ──────────────────────────────────────────────────────────────────
    K_BASE          = 5
    K_ALPHA         = 10
    EDGE_THRESHOLD  = 0.3
    REFINE_EVERY    = 2

    # ── Training ───────────────────────────────────────────────────────────────
    BATCH_SIZE      = 64
    EPOCHS          = 50
    LR              = 1e-3
    WEIGHT_DECAY    = 1e-4
    IMG_SIZE        = 224
    SEED            = 42
    NUM_WORKERS     = 2
    RESUME          = False

    # ── Device ─────────────────────────────────────────────────────────────────
    DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)
print(f"{'='*60}")
print(f"  TAL-GRN  |  Device: {cfg.DEVICE.upper()}")
print(f"{'='*60}")

# ──────────────────────────────────────────────────────────────────────────────
# CELL 3: Dataset
# ──────────────────────────────────────────────────────────────────────────────
class ImageDataset(Dataset):
    def __init__(self, csv_path, image_root, img_col, label_col,
                 transform=None, label_map=None, img_size=224):
        self.df         = pd.read_csv(csv_path)
        self.image_root = image_root
        self.img_col    = img_col
        self.label_col  = label_col
        self.transform  = transform
        self.img_size   = img_size

        if label_map is None:
            unique = sorted(self.df[label_col].unique())
            self.label_map = {v: i for i, v in enumerate(unique)}
        else:
            self.label_map = label_map

        self.labels = [self.label_map[l] for l in self.df[label_col]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rel_path = self.df.iloc[idx][self.img_col]
        img_path = os.path.join(self.image_root, rel_path)
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (self.img_size, self.img_size))
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label, img_path


def get_transforms(split="train"):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if split == "train":
        return transforms.Compose([
            transforms.Resize((cfg.IMG_SIZE + 32, cfg.IMG_SIZE + 32)),
            transforms.RandomCrop(cfg.IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def build_loaders():
    train_ds  = ImageDataset(cfg.TRAIN_CSV, cfg.IMAGE_ROOT,
                              cfg.IMG_COL, cfg.LABEL_COL,
                              get_transforms("train"), img_size=cfg.IMG_SIZE)
    label_map = train_ds.label_map
    cfg.NUM_CLASSES = len(label_map)

    val_ds   = ImageDataset(cfg.VAL_CSV,  cfg.IMAGE_ROOT,
                             cfg.IMG_COL, cfg.LABEL_COL,
                             get_transforms("val"), label_map, img_size=cfg.IMG_SIZE)
    test_ds  = ImageDataset(cfg.TEST_CSV, cfg.IMAGE_ROOT,
                             cfg.IMG_COL, cfg.LABEL_COL,
                             get_transforms("val"), label_map, img_size=cfg.IMG_SIZE)

    train_dl = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                          shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE,
                          shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)

    print(f"\n  Train: {len(train_ds):>6} | Val: {len(val_ds):>5} | Test: {len(test_ds):>5}")
    print(f"  Classes: {cfg.NUM_CLASSES}")
    return train_dl, val_dl, test_dl, label_map

# ──────────────────────────────────────────────────────────────────────────────
# CELL 4: CNN Backbone (Frozen)
# ──────────────────────────────────────────────────────────────────────────────
class FrozenBackbone(nn.Module):
    def __init__(self, arch="resnet50"):
        super().__init__()
        base = getattr(models, arch)(pretrained=True)
        self.features = nn.Sequential(*list(base.children())[:-1])
        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, x):
        out = self.features(x)
        return out.flatten(1)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 5: Adaptive Neighborhood MLP
# ──────────────────────────────────────────────────────────────────────────────
class AdaptiveNeighborhoodMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, h):
        return self.net(h).squeeze(-1)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 6: Edge Weight Predictor (EWP)
# ──────────────────────────────────────────────────────────────────────────────
class EdgeWeightPredictor(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim * 2 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, h_i, h_j, s_ij):
        e = torch.cat([h_i, h_j, s_ij], dim=-1)
        return F.softplus(self.net(e)).squeeze(-1)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 7: GraphSAGE-Style GNN Layer (AMP-safe)
# ──────────────────────────────────────────────────────────────────────────────
class TALGRNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W_m  = nn.Linear(in_dim, out_dim, bias=False)
        self.W_u  = nn.Linear(in_dim + out_dim, out_dim)
        self.act  = nn.GELU()
        self.norm = nn.LayerNorm(out_dim)
        self.res  = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()

    def forward(self, h, adj_sparse):
        h_proj = self.W_m(h)
        with torch.cuda.amp.autocast(enabled=False):
            m = torch.sparse.mm(adj_sparse.float(), h_proj.float())
        m = m.to(h.dtype)
        h_new = self.act(self.W_u(torch.cat([h, m], dim=-1)))
        h_new = self.norm(h_new)
        return h_new + self.res(h)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 8: Graph Construction Utilities
# ──────────────────────────────────────────────────────────────────────────────
def cosine_similarity_matrix(h):
    h_norm = F.normalize(h, p=2, dim=-1)
    return h_norm @ h_norm.T


def build_adaptive_graph(h, neighborhood_mlp, ewp, threshold=0.3):
    N = h.size(0)
    device = h.device

    S = cosine_similarity_matrix(h)

    with torch.no_grad():
        a = neighborhood_mlp(h)
    k_float = cfg.K_BASE + cfg.K_ALPHA * torch.sigmoid(a)
    k_int   = k_float.long().clamp(1, max(N - 1, 1))
    k_max   = int(k_int.max().item())

    S_no_self = S.clone()
    S_no_self.fill_diagonal_(-2.0)

    _, topk_idx = torch.topk(S_no_self, k_max, dim=1)

    k_range = torch.arange(k_max, device=device).unsqueeze(0)
    valid   = k_range < k_int.unsqueeze(1)

    row_idx = torch.arange(N, device=device).unsqueeze(1).expand(N, k_max)
    rows    = row_idx[valid]
    cols    = topk_idx[valid]

    h_i  = h[rows]
    h_j  = h[cols]
    s_ij = S[rows, cols].unsqueeze(-1)
    w    = ewp(h_i, h_j, s_ij)

    w = w.clamp(min=0.0, max=10.0)

    mask = w > threshold
    rows, cols, w = rows[mask], cols[mask], w[mask]

    self_idx = torch.arange(N, device=device)
    rows = torch.cat([rows, self_idx])
    cols = torch.cat([cols, self_idx])
    w    = torch.cat([w, torch.ones(N, device=device, dtype=w.dtype)])

    A_idx = torch.stack([rows, cols])
    A     = torch.sparse_coo_tensor(A_idx, w, (N, N)).coalesce()

    A_dense = A.to_dense()
    deg     = A_dense.sum(1)
    D_inv_sqrt = torch.zeros_like(deg)
    pos = deg > 0
    D_inv_sqrt[pos] = deg[pos].pow(-0.5)

    A_norm = D_inv_sqrt.unsqueeze(1) * A_dense * D_inv_sqrt.unsqueeze(0)
    A_norm = torch.nan_to_num(A_norm, nan=0.0, posinf=0.0, neginf=0.0)

    return A_norm.to_sparse()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 9: Full TAL-GRN Model
# ──────────────────────────────────────────────────────────────────────────────
class TALGRN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        d = cfg.FEATURE_DIM
        H = cfg.GNN_HIDDEN_DIM

        self.backbone         = FrozenBackbone(cfg.BACKBONE)
        self.neighborhood_mlp = AdaptiveNeighborhoodMLP(d)
        self.ewp              = EdgeWeightPredictor(d)

        dims = [d] + [H] * cfg.GNN_LAYERS
        self.gnn_layers = nn.ModuleList([
            TALGRNLayer(dims[i], dims[i+1]) for i in range(cfg.GNN_LAYERS)
        ])

        self.classifier = nn.Linear(H, num_classes)

    def forward(self, x, return_features=False, return_activations=False):
        h = self.backbone(x)

        A = build_adaptive_graph(
            h.detach(),
            self.neighborhood_mlp,
            self.ewp,
            threshold=cfg.EDGE_THRESHOLD
        )

        activations = []
        for layer in self.gnn_layers:
            h = layer(h, A)
            if return_activations:
                activations.append(h)

        logits = self.classifier(h)

        if return_features and return_activations:
            return logits, h, activations
        elif return_features:
            return logits, h
        return logits

# ──────────────────────────────────────────────────────────────────────────────
# CELL 10: Grad-CAM Implementation
# ──────────────────────────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        handle_fwd = self.target_layer.register_forward_hook(forward_hook)
        handle_bwd = self.target_layer.register_backward_hook(backward_hook)
        self.hook_handles = [handle_fwd, handle_bwd]

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

    def generate_cam(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)

        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()

        self.model.zero_grad()

        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        gradients = self.gradients.detach()
        activations = self.activations.detach()

        weights = torch.mean(gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam.squeeze().cpu().numpy(), class_idx


def overlay_cam_on_image(img_tensor, cam, alpha=0.5):
    """Overlay CAM on image tensor."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    img = img.permute(1, 2, 0).numpy()

    import cv2
    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
    cam_resized = np.uint8(255 * cam_resized)
    cam_resized = cv2.applyColorMap(cam_resized, cv2.COLORMAP_JET)
    cam_resized = cv2.cvtColor(cam_resized, cv2.COLOR_BGR2RGB)
    cam_resized = cam_resized / 255.0

    overlay = (1 - alpha) * img + alpha * cam_resized
    return np.clip(overlay, 0, 1)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 11: Metrics
# ──────────────────────────────────────────────────────────────────────────────
def compute_metrics(all_labels, all_preds, all_probs, num_classes, light=False):
    metrics = {}

    metrics["accuracy"] = accuracy_score(all_labels, all_preds)
    metrics["f1_macro"] = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    if light:
        for k in ["precision_macro", "recall_macro", "precision_weighted",
                  "recall_weighted", "f1_weighted", "roc_auc", "roc_auc_ovr", "roc_auc_ovo", "mcc"]:
            metrics[k] = float("nan")
        metrics["f1_per_class"] = np.array([])
        return metrics

    metrics["precision_macro"]    = precision_score(all_labels, all_preds, average="macro",    zero_division=0)
    metrics["recall_macro"]       = recall_score(   all_labels, all_preds, average="macro",    zero_division=0)
    metrics["precision_weighted"] = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    metrics["recall_weighted"]    = recall_score(   all_labels, all_preds, average="weighted", zero_division=0)
    metrics["f1_weighted"]        = f1_score(       all_labels, all_preds, average="weighted", zero_division=0)
    metrics["f1_per_class"]       = f1_score(       all_labels, all_preds, average=None,       zero_division=0)

    try:
        if num_classes == 2:
            metrics["roc_auc"]     = roc_auc_score(all_labels, all_probs[:, 1])
            metrics["roc_auc_ovr"] = float("nan")
            metrics["roc_auc_ovo"] = float("nan")
        else:
            y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
            metrics["roc_auc_ovr"] = roc_auc_score(y_bin, all_probs, average="macro", multi_class="ovr")
            metrics["roc_auc_ovo"] = roc_auc_score(y_bin, all_probs, average="macro", multi_class="ovo")
            metrics["roc_auc"]     = metrics["roc_auc_ovr"]
    except Exception:
        metrics["roc_auc"]     = float("nan")
        metrics["roc_auc_ovr"] = float("nan")
        metrics["roc_auc_ovo"] = float("nan")

    metrics["mcc"] = matthews_corrcoef(all_labels, all_preds)
    return metrics

# ──────────────────────────────────────────────────────────────────────────────
# CELL 12: Evaluation Loop
# ──────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, criterion, num_classes, split="Val", light=False):
    model.eval()
    total_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []
    all_image_paths = []

    bar = tqdm(loader, desc=f"  {split:>5}", leave=False,
               bar_format="{l_bar}{bar:30}{r_bar}")

    for imgs, labels, paths in bar:
        imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)

        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=1)
        all_probs.append(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_image_paths.extend(paths)

    avg_loss   = total_loss / len(loader.dataset)
    all_probs  = np.vstack(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)

    metrics = compute_metrics(all_labels, all_preds, all_probs, num_classes, light=light)
    metrics["loss"] = avg_loss

    return metrics, all_labels, all_preds, all_probs, all_image_paths

# ──────────────────────────────────────────────────────────────────────────────
# CELL 13: Pretty Printer
# ──────────────────────────────────────────────────────────────────────────────
def print_metrics(metrics, split="Val", epoch=None, num_classes=None, label_map=None):
    tag = f" [Epoch {epoch}] " if epoch else " "
    print(f"\n{'─'*60}")
    print(f"  {split.upper()}{tag}METRICS")
    print(f"{'─'*60}")
    print(f"  {'Loss':<30} {metrics['loss']:.4f}")
    print(f"  {'Accuracy':<30} {metrics['accuracy']*100:.2f}%")
    print(f"  {'Precision (Macro)':<30} {metrics['precision_macro']:.4f}")
    print(f"  {'Recall / Sensitivity (Macro)':<30} {metrics['recall_macro']:.4f}")
    print(f"  {'F1-Score (Macro)':<30} {metrics['f1_macro']:.4f}")
    print(f"  {'F1-Score (Weighted)':<30} {metrics['f1_weighted']:.4f}")
    print(f"  {'ROC-AUC (Macro OvR)':<30} {metrics['roc_auc_ovr']:.4f}")
    print(f"  {'ROC-AUC (Macro OvO)':<30} {metrics['roc_auc_ovo']:.4f}")
    print(f"  {'MCC':<30} {metrics['mcc']:.4f}")
    print(f"{'─'*60}")

    if num_classes and num_classes <= 30:
        inv_map = {v: k for k, v in label_map.items()} if label_map else {}
        print(f"  Per-Class F1:")
        for c, f1 in enumerate(metrics["f1_per_class"]):
            cname = inv_map.get(c, str(c))
            bar   = "█" * int(f1 * 20) + "░" * (20 - int(f1 * 20))
            print(f"    {cname:<20} [{bar}] {f1:.3f}")
    print()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 14: Checkpoint Utilities
# ──────────────────────────────────────────────────────────────────────────────
def save_checkpoint(state, path):
    torch.save(state, path)
    print(f"  ✔ Checkpoint saved → {path}")


def load_latest_checkpoint(model, optimizer, scheduler):
    ckpts = sorted(Path(cfg.CHECKPOINT_DIR).glob("epoch_*.pt"))
    if not ckpts:
        return 0, 0.0
    latest = ckpts[-1]
    print(f"  ↩ Resuming from {latest.name}")
    ck = torch.load(latest, map_location=cfg.DEVICE, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    if scheduler and "scheduler" in ck:
        scheduler.load_state_dict(ck["scheduler"])
    return ck["epoch"], ck.get("best_f1", 0.0)

# ──────────────────────────────────────────────────────────────────────────────
# CELL 15: Training Loop (with AMP)
# ──────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, epoch, scaler):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    bar = tqdm(loader, desc=f"  Epoch {epoch:>3}",
               bar_format="{l_bar}{bar:35}{r_bar}")

    for step, (imgs, labels, _) in enumerate(bar):
        imgs, labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        if not torch.isfinite(loss):
            print(f"  ⚠ non-finite loss at epoch {epoch} step {step} — batch skipped")
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds = logits.detach().argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc":  f"{100.*correct/max(total,1):.1f}%"
        })

    return total_loss / max(total, 1), correct / max(total, 1)


def train(model, train_dl, val_dl, label_map):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    start_epoch = 0
    best_f1     = 0.0

    if cfg.RESUME:
        start_epoch, best_f1 = load_latest_checkpoint(model, optimizer, scheduler)

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss":   [], "val_acc":   [],
        "val_f1":     [], "val_auc":   [], "val_mcc": []
    }

    print(f"\n{'='*60}")
    print(f"  Starting training from epoch {start_epoch+1} / {cfg.EPOCHS}")
    print(f"{'='*60}\n")

    for epoch in range(start_epoch + 1, cfg.EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(
            model, train_dl, optimizer, criterion, epoch, scaler
        )

        heavy = (epoch % 5 == 0)
        val_metrics, _, _, _, _ = evaluate(
            model, val_dl, criterion, cfg.NUM_CLASSES, "Val", light=not heavy
        )

        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_f1"].append(val_metrics["f1_macro"])
        history["val_auc"].append(val_metrics["roc_auc"])
        history["val_mcc"].append(val_metrics["mcc"])

        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]

        print(
            f"  Ep {epoch:>3}/{cfg.EPOCHS}  "
            f"TrLoss={tr_loss:.4f}  TrAcc={tr_acc*100:.1f}%  "
            f"ValLoss={val_metrics['loss']:.4f}  "
            f"ValF1={val_metrics['f1_macro']:.4f}  "
            f"ValAcc={val_metrics['accuracy']*100:.1f}%  "
            f"LR={lr_now:.2e}  "
            f"[{elapsed:.0f}s]"
        )

        is_best = val_metrics["f1_macro"] > best_f1
        if is_best:
            best_f1 = val_metrics["f1_macro"]
            save_checkpoint({
                "epoch": epoch, "model": model.state_dict(), "best_f1": best_f1
            }, os.path.join(cfg.CHECKPOINT_DIR, "best_model.pt"))
            print(f"  ★ New best model (F1={best_f1:.4f}) saved!")

        if epoch % 5 == 0 or epoch == cfg.EPOCHS:
            save_checkpoint({
                "epoch":     epoch,
                "model":     model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_f1":   best_f1,
                "metrics":   val_metrics,
            }, os.path.join(cfg.CHECKPOINT_DIR, f"epoch_{epoch:03d}.pt"))
            print_metrics(val_metrics, "Val", epoch, cfg.NUM_CLASSES, label_map)

    history_df = pd.DataFrame({
        'epoch': list(range(1, len(history['train_loss']) + 1)),
        'train_loss': history['train_loss'],
        'train_acc': history['train_acc'],
        'val_loss': history['val_loss'],
        'val_acc': history['val_acc'],
        'val_f1': history['val_f1'],
        'val_auc': history['val_auc'],
        'val_mcc': history['val_mcc'],
    })
    history_df.to_csv(os.path.join(cfg.CHECKPOINT_DIR, 'training_history.csv'), index=False)
    return history

# ──────────────────────────────────────────────────────────────────────────────
# CELL 16: Visualize Training Curves
# ──────────────────────────────────────────────────────────────────────────────
def plot_history(history, save_path=None):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("TAL-GRN Training History", fontsize=16, fontweight="bold")

    pairs = [
        ("Loss",     "train_loss", "val_loss",  "Train Loss", "Val Loss"),
        ("Accuracy", "train_acc",  "val_acc",   "Train Acc",  "Val Acc"),
        ("F1 Macro", None,         "val_f1",    None,         "Val F1"),
        ("ROC-AUC",  None,         "val_auc",   None,         "Val AUC"),
        ("MCC",      None,         "val_mcc",   None,         "Val MCC"),
    ]

    for ax, (title, tr_key, va_key, tr_label, va_label) in zip(axes.flatten(), pairs):
        epochs = list(range(1, len(history["val_loss"]) + 1))
        if tr_key and tr_key in history:
            ax.plot(epochs, history[tr_key], label=tr_label, color="#2196F3", linewidth=2)
        ax.plot(epochs, history[va_key], label=va_label, color="#FF5722", linewidth=2)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1, -1].set_visible(False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Plot saved → {save_path}")
    plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 17: Confusion Matrix (Fixed - with bold, spaced labels for 40 classes)
# ──────────────────────────────────────────────────────────────────────────────
def plot_confusion_matrix(all_labels, all_preds, label_map, save_path=None, figsize=(20, 18)):
    inv_map = {v: k for k, v in label_map.items()}
    class_names = [inv_map[i] for i in range(len(label_map))]
    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    n_classes = len(class_names)
    figsize = (max(20, n_classes * 0.8), max(18, n_classes * 0.7))

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Confusion Matrix (Counts)", "Confusion Matrix (Normalized)"],
        ["d", ".2f"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5,
                    annot_kws={'size': 6 if n_classes > 30 else 8, 'weight': 'bold'},
                    cbar_kws={'shrink': 0.8})
        ax.set_title(title, fontweight="bold", fontsize=14)
        ax.set_xlabel("Predicted", fontsize=12, fontweight="bold")
        ax.set_ylabel("True", fontsize=12, fontweight="bold")
        ax.tick_params(axis='x', rotation=90, labelsize=8, width=2, length=4)
        ax.tick_params(axis='y', labelsize=8, width=2, length=4)

        for label in ax.get_xticklabels():
            label.set_fontweight('bold')
        for label in ax.get_yticklabels():
            label.set_fontweight('bold')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  Confusion matrix saved → {save_path}")
    plt.show()

    print("\n  Per-Class Accuracy (Diagonal of normalized confusion matrix):")
    print("  " + "-" * 50)
    for i, (name, acc) in enumerate(zip(class_names, cm_norm.diagonal())):
        print(f"    {name:<30}: {acc:.3f}")

# ──────────────────────────────────────────────────────────────────────────────
# CELL 18: Per-Class Metrics Bar Chart
# ──────────────────────────────────────────────────────────────────────────────
def plot_per_class_metrics(all_labels, all_preds, label_map, save_path=None):
    inv_map   = {v: k for k, v in label_map.items()}
    class_names = [inv_map[i] for i in range(len(label_map))]

    prec  = precision_score(all_labels, all_preds, average=None, zero_division=0)
    rec   = recall_score(   all_labels, all_preds, average=None, zero_division=0)
    f1    = f1_score(       all_labels, all_preds, average=None, zero_division=0)

    x = np.arange(len(class_names))
    w = 0.25
    fig, ax = plt.subplots(figsize=(max(15, len(class_names) * 0.6), 7))
    ax.bar(x - w, prec, w, label="Precision", color="#42A5F5")
    ax.bar(x,     rec,  w, label="Recall",    color="#66BB6A")
    ax.bar(x + w, f1,   w, label="F1",        color="#FFA726")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=90, ha="right", fontweight="bold", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-Class Precision / Recall / F1", fontweight="bold", fontsize=14)
    ax.legend(fontsize=12)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis='y', labelsize=10)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 18.5: ROC Curves (Multi-class)
# ──────────────────────────────────────────────────────────────────────────────
def plot_roc_curves(all_labels, all_probs, label_map, save_path=None):
    """Plot ROC curves for each class in multi-class classification."""
    n_classes = len(label_map)
    inv_map = {v: k for k, v in label_map.items()}
    class_names = [inv_map[i] for i in range(n_classes)]

    y_bin = label_binarize(all_labels, classes=list(range(n_classes)))

    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), all_probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    fig, ax = plt.subplots(figsize=(12, 10))

    ax.plot(fpr["micro"], tpr["micro"],
            label=f'micro-average ROC (AUC = {roc_auc["micro"]:.3f})',
            color='deeppink', linestyle=':', linewidth=4)

    colors = plt.cm.tab20(np.linspace(0, 1, n_classes))
    for i, color in zip(range(n_classes), colors):
        ax.plot(fpr[i], tpr[i], color=color, lw=2,
                label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    ax.set_title('ROC Curves - Multi-class Classification', fontsize=16, fontweight='bold')
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  ROC curves saved → {save_path}")
    plt.show()

    return fpr, tpr, roc_auc

# ──────────────────────────────────────────────────────────────────────────────
# CELL 18.6: MCC Curve Over Epochs
# ──────────────────────────────────────────────────────────────────────────────
def plot_mcc_curve(history, save_path=None):
    """Plot MCC values over training epochs."""
    epochs = list(range(1, len(history['val_mcc']) + 1))

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(epochs, history['val_mcc'], marker='o', markersize=4,
            color='#9C27B0', linewidth=2, label='Validation MCC')

    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

    ax.set_xlabel('Epoch', fontsize=14, fontweight='bold')
    ax.set_ylabel('Matthews Correlation Coefficient (MCC)', fontsize=14, fontweight='bold')
    ax.set_title('MCC Curve Over Training Epochs', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    max_mcc = max(history['val_mcc'])
    max_epoch = history['val_mcc'].index(max_mcc) + 1
    ax.annotate(f'Best MCC: {max_mcc:.4f}\nEpoch {max_epoch}',
                xy=(max_epoch, max_mcc),
                xytext=(max_epoch + 2, max_mcc - 0.05),
                fontsize=10,
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  MCC curve saved → {save_path}")
    plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 18.7: ROC Curve over Epochs (Training Progress)
# ──────────────────────────────────────────────────────────────────────────────
def plot_roc_over_epochs(history, save_path=None):
    """Plot ROC-AUC values over training epochs."""
    epochs = list(range(1, len(history['val_auc']) + 1))

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(epochs, history['val_auc'], marker='s', markersize=4,
            color='#FF6B6B', linewidth=2, label='Validation ROC-AUC')

    ax.set_xlabel('Epoch', fontsize=14, fontweight='bold')
    ax.set_ylabel('ROC-AUC Score', fontsize=14, fontweight='bold')
    ax.set_title('ROC-AUC Curve Over Training Epochs', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim([0, 1.05])

    max_auc = max(history['val_auc'])
    max_epoch = history['val_auc'].index(max_auc) + 1
    ax.annotate(f'Best AUC: {max_auc:.4f}\nEpoch {max_epoch}',
                xy=(max_epoch, max_auc),
                xytext=(max_epoch + 2, max_auc - 0.05),
                fontsize=10,
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"  ROC-AUC over epochs saved → {save_path}")
    plt.show()

# ──────────────────────────────────────────────────────────────────────────────
# CELL 19: Grad-CAM Visualization for One Accurate Image Per Class
# ──────────────────────────────────────────────────────────────────────────────
def visualize_gradcam_per_class(model, test_dl, label_map, save_dir):
    """Generate Grad-CAM visualizations for one accurately predicted image per class."""
    model.eval()

    os.makedirs(save_dir, exist_ok=True)

    target_layer = model.backbone.features[-2]
    gradcam = GradCAM(model, target_layer)

    inv_map = {v: k for k, v in label_map.items()}
    class_names = [inv_map[i] for i in range(len(label_map))]

    class_examples = {}
    class_image_paths = {}

    print(f"\n  Collecting accurate predictions per class...")

    with torch.no_grad():
        for imgs, labels, paths in tqdm(test_dl, desc="  Finding examples"):
            imgs = imgs.to(cfg.DEVICE)
            labels_np = labels.cpu().numpy()

            logits = model(imgs)
            probs = F.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=1).cpu().numpy()

            for i in range(len(labels_np)):
                true_label = labels_np[i]
                pred_label = preds[i]

                if true_label == pred_label and true_label not in class_examples:
                    class_examples[true_label] = (imgs[i:i+1], pred_label, probs[i:i+1])
                    class_image_paths[true_label] = paths[i]

                    if len(class_examples) >= len(class_names):
                        break

            if len(class_examples) >= len(class_names):
                break

    print(f"\n  Generating Grad-CAM visualizations for {len(class_examples)} classes...")

    n_cols = 5
    n_rows = (len(class_examples) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes]

    for idx, (class_id, (img_tensor, pred_label, probs)) in enumerate(sorted(class_examples.items())):
        if idx >= len(axes):
            break

        ax = axes[idx]

        cam, _ = gradcam.generate_cam(img_tensor, class_idx=class_id)
        overlay_img = overlay_cam_on_image(img_tensor[0], cam, alpha=0.5)

        ax.imshow(overlay_img)
        class_name = class_names[class_id]
        confidence = probs[0, class_id].item() * 100
        ax.set_title(f"{class_name}\nConf: {confidence:.1f}%",
                     fontweight="bold", fontsize=10)
        ax.axis('off')

        save_path = os.path.join(save_dir, f"gradcam_class_{class_id}_{class_name}.png")
        plt.imsave(save_path, overlay_img)

    for idx in range(len(class_examples), len(axes)):
        axes[idx].axis('off')

    plt.suptitle("Grad-CAM Visualizations - One Accurate Prediction Per Class",
                 fontsize=16, fontweight="bold")
    plt.tight_layout()

    combined_path = os.path.join(save_dir, "gradcam_all_classes.png")
    plt.savefig(combined_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"  Grad-CAM visualizations saved to: {save_dir}")
    print(f"  Combined image saved to: {combined_path}")

    gradcam.remove_hooks()
    return class_examples

# ──────────────────────────────────────────────────────────────────────────────
# CELL 20: Final Evaluation on Test Set with Grad-CAM and Curves
# ──────────────────────────────────────────────────────────────────────────────
def final_test(model, test_dl, label_map, run_seed, history=None):
    criterion = nn.CrossEntropyLoss()

    best_ck = os.path.join(cfg.CHECKPOINT_DIR, "best_model.pt")
    if os.path.exists(best_ck):
        ck = torch.load(best_ck, map_location=cfg.DEVICE)
        model.load_state_dict(ck["model"])
        print(f"  Loaded best model (epoch {ck['epoch']}, F1={ck['best_f1']:.4f})")

    test_metrics, all_labels, all_preds, all_probs, all_paths = evaluate(
        model, test_dl, criterion, cfg.NUM_CLASSES, "Test"
    )
    print_metrics(test_metrics, "TEST", num_classes=cfg.NUM_CLASSES, label_map=label_map)

    inv_map     = {v: k for k, v in label_map.items()}
    class_names = [inv_map[i] for i in range(cfg.NUM_CLASSES)]
    print("\n  FULL CLASSIFICATION REPORT")
    print("  " + "─"*58)
    report = classification_report(
        all_labels, all_preds, target_names=class_names, zero_division=0
    )
    for line in report.split("\n"):
        print("  " + line)

    plot_dir = os.path.join(cfg.CHECKPOINT_DIR, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    plot_confusion_matrix(
        all_labels, all_preds, label_map,
        save_path=os.path.join(plot_dir, "confusion_matrix.png")
    )
    plot_per_class_metrics(
        all_labels, all_preds, label_map,
        save_path=os.path.join(plot_dir, "per_class_metrics.png")
    )

    try:
        plot_roc_curves(
            all_labels, all_probs, label_map,
            save_path=os.path.join(plot_dir, "roc_curves.png")
        )
    except Exception as e:
        print(f"  ⚠ Could not plot ROC curves: {e}")

    if history is not None:
        plot_mcc_curve(
            history,
            save_path=os.path.join(plot_dir, "mcc_curve.png")
        )
        plot_roc_over_epochs(
            history,
            save_path=os.path.join(plot_dir, "roc_auc_over_epochs.png")
        )

    gradcam_dir = os.path.join(plot_dir, "gradcam")
    visualize_gradcam_per_class(model, test_dl, label_map, gradcam_dir)

    return test_metrics

# ════════════════════════════════════════════════════════════════════════════
#  CELL 21: STATISTICAL EXPERIMENT — 5 RANDOM SEEDS
# ════════════════════════════════════════════════════════════════════════════
SEEDS = [42, 0, 1, 7, 123]
all_run_metrics = []

for run_idx, seed in enumerate(SEEDS):
    print(f"\n{'#'*60}")
    print(f"#  RUN {run_idx+1}/{len(SEEDS)}  |  SEED = {seed}")
    print(f"{'#'*60}")

    cfg.SEED          = seed
    cfg.RESUME        = False
    cfg.NUM_CLASSES   = None
    cfg.CHECKPOINT_DIR = f"/kaggle/working/TAL_GRN_checkpoints/seed_{seed}"
    os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)

    set_seed(seed)

    print(f"\n{'='*60}")
    print(f"  TAL-GRN  ·  Long-Tailed Image Classification")
    print(f"{'='*60}")
    print(f"  Device    : {cfg.DEVICE}")
    print(f"  Backbone  : {cfg.BACKBONE}")
    print(f"  Epochs    : {cfg.EPOCHS}")
    print(f"  Batch     : {cfg.BATCH_SIZE}")
    print(f"  GNN Layers: {cfg.GNN_LAYERS}")
    print(f"  K_base    : {cfg.K_BASE}  K_alpha: {cfg.K_ALPHA}")
    print(f"  Seed      : {seed}")

    train_dl, val_dl, test_dl, label_map = build_loaders()

    model = TALGRN(num_classes=cfg.NUM_CLASSES).to(cfg.DEVICE)
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")

    history = train(model, train_dl, val_dl, label_map)

    plot_history(history)

    print(f"\n{'='*60}")
    print(f"  FINAL TEST EVALUATION  [Seed={seed}]")
    print(f"{'='*60}")
    test_metrics = final_test(model, test_dl, label_map, seed, history=history)

    all_run_metrics.append({"seed": seed, **test_metrics})

    print(f"\n  ✓ Run {run_idx+1}/{len(SEEDS)} complete")
    print(f"    Seed={seed}  "
          f"Acc={test_metrics['accuracy']*100:.2f}%  "
          f"F1={test_metrics['f1_macro']:.4f}  "
          f"AUC-OvR={test_metrics['roc_auc_ovr']:.4f}  "
          f"MCC={test_metrics['mcc']:.4f}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ════════════════════════════════════════════════════════════════════════════
#  AGGREGATE STATISTICAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════
METRIC_KEYS = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
    "roc_auc_ovr",
    "roc_auc_ovo",
    "mcc",
    "loss",
]

print(f"\n\n{'='*70}")
print(f"  STATISTICAL SUMMARY ACROSS {len(SEEDS)} SEEDS")
print(f"  Seeds used : {SEEDS}")
print(f"{'='*70}")

seed_header = "".join([f"  {'Seed-'+str(m['seed']):<12}" for m in all_run_metrics])
print(f"\n  {'Metric':<22}{seed_header}  {'Mean':<10}  {'Median':<10}  {'Std':<10}  {'Min':<10}  {'Max':<10}")
print("  " + "─" * 110)

for key in METRIC_KEYS:
    raw_vals = [m.get(key, float("nan")) for m in all_run_metrics]
    valid    = np.array([v for v in raw_vals if not np.isnan(v)])

    is_acc = key == "accuracy"

    def fmt(v):
        if np.isnan(v):
            return f"{'N/A':<12}"
        return f"{v*100:.2f}%{'':<5}" if is_acc else f"{v:.4f}{'':<6}"

    per_seed_str = "".join([f"  {fmt(v)}" for v in raw_vals])
    print(f"  {key.upper():<22}{per_seed_str}", end="")

    if len(valid) > 0:
        mn, md, sd, lo, hi = (np.mean(valid), np.median(valid),
                               np.std(valid),  np.min(valid), np.max(valid))
        if is_acc:
            print(f"  {mn*100:.2f}%{'':4}  {md*100:.2f}%{'':4}  "
                  f"{sd*100:.2f}%{'':4}  {lo*100:.2f}%{'':4}  {hi*100:.2f}%")
        else:
            print(f"  {mn:.4f}{'':4}  {md:.4f}{'':4}  "
                  f"{sd:.4f}{'':4}  {lo:.4f}{'':4}  {hi:.4f}")
    else:
        print("  N/A")

print(f"\n{'='*70}")
print(f"  All {len(SEEDS)} seed runs complete.")
print(f"{'='*70}\n")